# The PageRank Algorithm

**Seong Cho** · ACME Volume 1, Lab 13

**Lab objective:** model a network as a graph and implement the PageRank algorithm based on this model, then use PageRank to predict the rankings of sports teams.

<sub>Lab specification from the *Volume 1* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from the spec; the notes are condensed from it. Solutions are my own.</sub>

In [1]:
import numpy as np
from scipy.sparse import dok_matrix
from scipy import linalg as la
import networkx as nx

As of 2013, the PageRank algorithm is one of over 200 algorithms that Google uses to determine the rank, or relative importance, of a webpage. Named for Larry Page, cofounder of Google, it ranks pages based on how many other pages link to them.

## The Internet as a Graph

PageRank models the internet with a directed graph: each webpage is a node, and there is an edge from node $i$ to node $j$ if page $i$ links to page $j$. Let $\mathrm{In}(i)$ be the set of nodes with an arrow to node $i$ and $\mathrm{Out}(i)$ the set of nodes with an arrow from node $i$.

The spec's Figure 14.1 is a graph of 8 webpages with edges

$$0\to7,\quad 1\to0,\quad 3\to0,\ 3\to2,\ 3\to6,\quad 4\to0,\ 4\to5,\ 4\to6,\quad 5\to0,\ 5\to6,\quad 6\to0,\quad 7\to0,$$

so $\mathrm{In}(0) = \{1, 3, 4, 5, 6, 7\}$ and $\mathrm{Out}(0) = \{7\}$. A link from a more important page counts more than one from a less important page: node 0 should rank very high because every other node links to it, and node 7 fairly high because node 0 links to it, even though node 0 is the only node to do so.

## The PageRank Algorithm

A surfer chooses a starting page at random; then, at page $i$, they randomly select a page from $\mathrm{Out}(i)$ to visit next. If the internet has $N$ pages and $p_i(t)$ is the likelihood that the surfer is on page $i$ at time $t$, then

$$p_i(0) = \frac{1}{N}, \qquad p_i(t+1) = \sum_{j \in \mathrm{In}(i)} \frac{p_j(t)}{|\mathrm{Out}(j)|}. \tag{14.1}$$

For example, in Figure 14.1, $N = 8$ and $p_6(t+1) = \frac{p_3(t)}{3} + \frac{p_4(t)}{3} + \frac{p_5(t)}{2}$.

**Pages with no outbound links.** A node with no outbound links, such as node 2 in Figure 14.1, is a *sink*; a surfer who reaches it would stay there forever. More realistically they would pick another page at random and begin surfing again, so the model replaces each sink with a node linking to every page. In Figure 14.2, a link from page 2 to every other page has been added.

**Adding boredom.** The surfer stays interested at each step with probability $d$, the *damping factor*, and gets bored and picks a new page at random with probability $1 - d$:

$$p_i(0) = \frac{1}{N}, \qquad p_i(t+1) = d \sum_{j \in \mathrm{In}(i)} \frac{p_j(t)}{|\mathrm{Out}(j)|} + \frac{1-d}{N}. \tag{14.2}$$

## Matrix Form of the PageRank Algorithm

In matrix form, (14.2) is

$$\mathbf{p}(0) = \frac{1}{N}\mathbf{1}, \qquad \mathbf{p}(t+1) = dK\mathbf{p}(t) + \frac{1-d}{N}\mathbf{1}, \tag{14.3}$$

where $\mathbf{p}(t) = (p_1(t), \ldots, p_N(t))^T$, $\mathbf{1}$ is a vector of $N$ ones, and

$$K_{ij} = \begin{cases} \dfrac{1}{|\mathrm{Out}(j)|} & \text{if } j \text{ links to } i \\ 0 & \text{otherwise.} \end{cases}$$

The rank of page $i$ is $p_i = \lim_{t\to\infty} p_i(t)$. For those familiar with Markov chains, (14.3) defines a Markov chain, and the page ranks are its steady state.

## Implementation in Python

The adjacency matrix $A$ of a directed graph has $A_{ij} = 1$ if there is an edge from node $i$ to node $j$, and $A_{ij} = 0$ otherwise. For Figure 14.1:

In [2]:
A = np.array([[ 0, 0, 0, 0, 0, 0, 0, 1],
              [ 1, 0, 0, 0, 0, 0, 0, 0],
              [ 0, 0, 0, 0, 0, 0, 0, 0],
              [ 1, 0, 1, 0, 0, 0, 1, 0],
              [ 1, 0, 0, 0, 0, 1, 1, 0],
              [ 1, 0, 0, 0, 0, 0, 1, 0],
              [ 1, 0, 0, 0, 0, 0, 0, 0],
              [ 1, 0, 0, 0, 0, 0, 0, 0]])

## Problem 1: Adjacency matrix from a file

> **Problem 1.** Write a function that creates an adjacency matrix from a file. The function should accept a filename, and an integer $N$ that represents the number of nodes in the graph described by the datafile. Return the adjacency matrix as a SciPy sparse `dok_matrix`.
>
> Hints:
> 1. The file `matrix.txt` included with this lab describes the matrix in Figure 14.1 and has the adjacency matrix $A$ given above. You may use it to test your function.
> 2. You can open a file in Python using the `with` syntax. Then, you can iterate through the lines using a `for` loop. Here is an example.
>
>    ```python
>    # Open 'matrix.txt' for read-only
>    with open('./matrix.txt', 'r') as myfile:
>        for line in myfile:
>            print line
>    ```
> 3. Here is an example of how to process a line of the form in datafile.
>
>    ```python
>    >>> line = '0\t4\n'
>    # strip() removes trailing whitespace from a line.
>    # split() returns a list of the space-separated pieces of the line.
>    >>> line.strip().split()
>    ['0', '4']
>    ```
> 4. Rather than testing for lines of `matrix.txt` that contain comments, put all your string operations in a `try` block with an `except` block following.

(Hint 2's `print line` is Python 2 syntax.) Hint 3, runnable:

In [3]:
line = '0\t4\n'
line.strip().split()

['0', '4']

> **Note.** It makes sense to initialize $A$ as a sparse matrix, since $A$ is mostly zeros. To make the coding easier, the rest of the lab codes the algorithms with non-sparse matrices, so cast an adjacency matrix from Problem 1 with `toarray()` before passing it to the other functions (for example, `test_calculateK(A.toarray(), N)`). In the real world, sparse adjacency matrices are generally much more time-efficient than dense ones.

In [4]:
def to_matrix(filename, n):
    """Return the nxn adjacency matrix described by datafile.

    Parameters:
        datafile (str): The name of a .txt file describing a directed graph.
        Lines describing edges should have the form '<from node>\t<to node>\n'.
        The file may also include comments.
    n (int): The number of nodes in the graph described by datafile

    Returns:
        A SciPy sparse dok_matrix.
    """
    M = dok_matrix((n,n)) # Create an empty matrix
    with open(filename, 'r') as myfile:
        for line in myfile:
            try:
                i = line.strip().split() # Process the line
                M[int(i[0]), int(i[1])] = 1 # Set the value of the adjacency matrix
            except ValueError as e:
                continue
    return M

`matrix.txt` starts with a `From Node	To Node` header line, which the `try`/`except` skips. The result matches the spec's $A$:

In [5]:
print(open('matrix.txt').read())
M = to_matrix('matrix.txt', 8)
print(type(M).__name__, M.shape, M.nnz, "edges")
assert np.array_equal(M.toarray(), A)

From Node	To Node
0	7
1	0
3	0
3	2
3	6
4	0
4	5
4	6
5	0
5	6
6	0
7	0
dok_matrix (8, 8) 12 edges


## Computing $K$

A good strategy for computing $K$ is to write

$$K = \left(D^{-1}A\right)^T,$$

where $D$ is the diagonal matrix with $D_{jj} = |\mathrm{Out}(j)|$, after modifying $A$ so that rows corresponding to sinks have all ones instead of all zeros. For Figure 14.2 the modified adjacency matrix and $K$ are below. $D$ is the vector of row sums of the modified $A$; it is better practice to store only the diagonal of $D$ and divide with array broadcasting. The transpose is needed because $D^{-1}A$ is *row stochastic* (its rows sum to 1), but we need to multiply by a *column stochastic* matrix (columns summing to 1).

In [6]:
Am = np.array([[ 0, 0, 0, 0, 0, 0, 0, 1],
               [ 1, 0, 0, 0, 0, 0, 0, 0],
               [ 1, 1, 1, 1, 1, 1, 1, 1],
               [ 1, 0, 1, 0, 0, 0, 1, 0],
               [ 1, 0, 0, 0, 0, 1, 1, 0],
               [ 1, 0, 0, 0, 0, 0, 1, 0],
               [ 1, 0, 0, 0, 0, 0, 0, 0],
               [ 1, 0, 0, 0, 0, 0, 0, 0]])

K = np.array([[ 0 , 1 , 1./8, 1./3, 1./3, 1./2, 1 , 1 ],
              [ 0 , 0 , 1./8, 0 , 0 , 0 , 0 , 0 ],
              [ 0 , 0 , 1./8, 1./3, 0 , 0 , 0 , 0 ],
              [ 0 , 0 , 1./8, 0 , 0 , 0 , 0 , 0 ],
              [ 0 , 0 , 1./8, 0 , 0 , 0 , 0 , 0 ],
              [ 0 , 0 , 1./8, 0 , 1./3, 0 , 0 , 0 ],
              [ 0 , 0 , 1./8, 1./3, 1./3, 1./2, 0 , 0 ],
              [ 1 , 0 , 1./8, 0 , 0 , 0 , 0 , 0 ]])

## Problem 2: The matrix $K$

> **Problem 2.** Write a function that computes and returns the $K$ matrix given an adjacency matrix.
> 1. Compute the diagonal matrix $D$.
> 2. Compute the modified adjacency matrix where the rows corresponding to sinks all have ones instead of zeros.
> 3. Compute $K$ using array broadcasting.

In [7]:
def calculateK(A,N):
    """Compute the matrix K as described in the lab.

    Parameters:
        A (ndarray): adjacency matrix of an array
        N (int): the datasize of the array

    Returns:
        K (ndarray)
    """
    # Changed: work on a copy. The sink rows were overwritten in place, which
    # also changed the caller's adjacency matrix (iter_solve()/eig_solve() pass
    # a view of 'adj'), so reusing that matrix afterwards gave wrong steady states.
    A = A.copy()
    # Compute the modified adjacency matrix for the sinks
    for i,r in enumerate(A):
        if np.allclose(r, np.zeros(N)):
            A[i] = np.ones(N)
    #Compute the diagonal matrix D
    D = np.array([sum(r) for r in A])
    # Compute K using array broadcasting
    return (A.T/D)

Checks against the spec's $K$, on the integer array and on the sparse matrix from Problem 1; the input is left unchanged, and the columns of $K$ sum to 1.

In [8]:
A_in = to_matrix('matrix.txt', 8).toarray()
K1 = calculateK(A_in, 8)
print(np.round(K1, 4))
assert np.allclose(K1, K) and np.allclose(calculateK(A, 8), K)
assert np.allclose(K1.sum(axis=0), 1)
assert np.array_equal(A_in, A)          # The adjacency matrix is not modified.

[[0.     1.     0.125  0.3333 0.3333 0.5    1.     1.    ]
 [0.     0.     0.125  0.     0.     0.     0.     0.    ]
 [0.     0.     0.125  0.3333 0.     0.     0.     0.    ]
 [0.     0.     0.125  0.     0.     0.     0.     0.    ]
 [0.     0.     0.125  0.     0.     0.     0.     0.    ]
 [0.     0.     0.125  0.     0.3333 0.     0.     0.    ]
 [0.     0.     0.125  0.3333 0.3333 0.5    0.     0.    ]
 [1.     0.     0.125  0.     0.     0.     0.     0.    ]]


## Solving for the Page Ranks

There are several ways to solve for $\lim_{t\to\infty}\mathbf{p}(t)$.

**Algebraic method.** For those familiar with Markov chains, one possibility is to assume the modified chain has a steady state $\mathbf{p}$ and solve for it algebraically:

$$(I - dK)\mathbf{p} = \frac{1-d}{N}\mathbf{1}. \tag{14.4}$$

SciPy's solver gives the page ranks of the network in Figure 14.2:

In [9]:
I = np.eye(8)
d = .85
p_solve = la.solve(I-d*K, ((1-d)/8)*np.ones(8))
p_solve

array([0.43869288, 0.02171029, 0.02786154, 0.02171029, 0.02171029,
       0.02786154, 0.04585394, 0.39459924])

As expected, node 0 has the highest rank, approximately .44. Node 7 ranks higher than node 6, even though $|\mathrm{In}(7)| = 1$ and $|\mathrm{In}(6)| = 3$, because node 7's single in-edge comes from a node with a very high rank (node 0).

**Iterative method.** Solving (14.4) is feasible for our small working example but not efficient for very large systems. Instead, start with a guess for $\mathbf{p}(0)$ and iterate (14.3) until $\|\mathbf{p}(t) - \mathbf{p}(t-1)\|$ is sufficiently small; at that point we assume the steady state has been reached.

## Problem 3: The iterative method

> **Problem 3.** Implement a function that uses the iterative method to find the steady state of the PageRank algorithm. You function should accept an adjacency matrix $A$, an integer $N$ that defaults to `None`, the damping factor $d$ that defaults to 0.85, and a tolerance `tol` that defaults to `1E-5`. Return the approximation to the steady state as a float. When the argument $N$ is not `None`, work with only the upper $N \times N$ portion of the array `adj`. Test your function against the example datafile that accompanies this lab.
>
> Hints:
> 1. Try making your initial guess for $\mathbf{p}(0)$ a random vector.
> 2. NumPy can do unexpected things with the dimensions when performing matrix-vector multiplication. When debugging, check at each iteration that all arrays have the dimensions you expect.

In [10]:
def iter_solve(adj, N=None, d=.85, tol=1E-5):
    """Return the page ranks of the network described by 'adj'.
    Iterate through the PageRank algorithm until the error is less than 'tol'.

    Parameters:
        adj (ndarray): The adjacency matrix of a directed graph.
        N (int): Restrict the computation to the first 'N' nodes of the graph.
            If N is None (default), use the entire matrix.
        d (float): The damping factor, a float between 0 and 1.
        tol (float): Stop iterating when the change in approximations to the
            solution is less than 'tol'.

    Returns:
        The approximation to the steady state.
    """
    # Handle the case when N=None
    if N == None:
        N =len(adj)
    # Work with the upper N X N matrix
    A = adj[:N, :N]
    K = calculateK(A, N)
    # Initialize p0 as a random vector
    p0 = np.random.random(N)
    # Calculate p1 using equation 14.3
    p1 = d*K@p0 + (1-d)/N * np.ones(N)
    # Continue with the iteration until within tolerance
    while la.norm(p1-p0) > tol:
        p0 = p1
        p1 = d*K@p0 + (1-d)/N * np.ones(N)

    return p1

On the example datafile, the iteration agrees with the algebraic solution (14.4) to within the tolerance. With $N = 5$ only the subgraph on nodes 0 to 4 is used (there, nodes 0 and 2 are both sinks); the reference for it is built independently of `calculateK()`.

In [11]:
np.random.seed(13)
A_in = to_matrix('matrix.txt', 8).toarray()
p_iter = iter_solve(A_in)
print(p_iter)
assert np.allclose(p_iter, p_solve, atol=1e-4)

def ref_pagerank(adj, d=.85):
    # Independent reference: sinks link everywhere, then solve (14.4).
    Am = adj.astype(float).copy()
    Am[Am.sum(axis=1) == 0] = 1
    Kr = (Am / Am.sum(axis=1)[:,None]).T
    n = len(adj)
    return la.solve(np.eye(n) - d*Kr, (1-d)/n*np.ones(n))

p5 = iter_solve(A_in, N=5)
print(p5)
assert np.allclose(p5, ref_pagerank(A[:5,:5]), atol=1e-4)
assert np.array_equal(A_in, A)

[0.43871163 0.02171029 0.02786154 0.02171029 0.02171029 0.02786154
 0.04585394 0.39461299]
[0.41395067 0.13246273 0.18876031 0.13246273 0.13246273]


## The Eigenvalue Method

Let $E$ be the $N \times N$ matrix of ones; then $E\mathbf{p}(t) = \mathbf{1}$, since the entries of $\mathbf{p}(t)$ sum to 1. Hence (14.3) becomes

$$\mathbf{p}(t+1) = \left(dK + \frac{1-d}{N}E\right)\mathbf{p}(t).$$

Writing $B = dK + \frac{1-d}{N}E$, this is $\mathbf{p}(t+1) = B\mathbf{p}(t)$, so the steady state is an eigenvector of $B$ for the eigenvalue 1. The columns of $B$ sum to 1 and its entries are strictly positive, so by the Perron-Frobenius theorem 1 is the unique eigenvalue of $B$ of largest magnitude, and the corresponding eigenvector is unique. The iterative method above is then just the power method for the dominant eigenvector, introduced in the lab on eigensolvers. SciPy's eigenvalue solvers can also compute $\mathbf{p}$.

## Problem 4: The eigenvalue method

> **Problem 4.** Implement a function that uses the eigenvalue method to find the steady state of the PageRank algorithm. Your function should accept an adjacency matrix $A$, an integer $N$ that defaults to `None`, and the damping factor $d$ that defaults to 0.85. Return the approximation to the steady state as a float.

In [12]:
def eig_solve(adj, N=None, d=.85):
    """Return the page ranks of the network described by 'adj'. Use SciPy's
    eigenvalue solver to calculate the steady state of the PageRank algorithm

    Parameters:
        adj (ndarray): The adjacency matrix of a directed graph.
        N (int): Restrict the computation to the first 'N' nodes of the graph.
            If N is None (default), use the entire matrix.
        d (float): The damping factor, a float between 0 and 1.
        tol (float): Stop iterating when the change in approximations to the
            solution is less than 'tol'.

    Returns:
        The approximation to the steady state.
    """
    # Same initial process as prob 3
    if N == None:
        N =len(adj)
    A = adj[:N, :N]
    K = calculateK(A, N)
    # Create E a matrix of ones
    E = np.ones((N,N))
    # Compute B using the formula given
    B = d*K + (1-d)/N * E
    # Compute the eigen values/vectors and get the vector that corresponds with 1
    val, vec = la.eig(B)
    i = np.argmax(val)
    p = vec[:,i]
    # Normalize
    # Changed: take the real part. la.eig() returns complex arrays, and the
    # spec asks for the steady state as floats.
    return (p / np.sum(p)).real

`la.eig()` does not return the eigenvalues in any particular order, so the code picks the eigenvalue 1 with `np.argmax()` rather than assuming it comes first (the 2018 grader's feedback on an earlier version was "eigenvalues do not come sorted"). Checks against three independent computations: the algebraic solution, the dominant eigenvector of $B$ from `np.linalg.eig()` built from the spec's $K$, and NetworkX's `pagerank()`.

In [13]:
A_in = to_matrix('matrix.txt', 8).toarray()
p_eig = eig_solve(A_in)
print(p_eig, p_eig.dtype)

B = .85*K + .15/8*np.ones((8,8))
vals, vecs = np.linalg.eig(B)
print("eigenvalues of B:", np.round(vals, 4))     # la.eig() promises no order.
j = np.argmin(np.abs(vals - 1))
p_np = np.real(vecs[:,j] / vecs[:,j].sum())

G = nx.from_numpy_array(A_in, create_using=nx.DiGraph)
pr = nx.pagerank(G, alpha=0.85, tol=1e-10, max_iter=1000)
p_nx = np.array([pr[i] for i in range(8)])

assert np.isrealobj(p_eig) and np.isclose(p_eig.sum(), 1)
assert np.allclose(p_eig, p_solve) and np.allclose(p_eig, p_np) and np.allclose(p_eig, p_nx)
assert np.allclose(eig_solve(A_in, N=5), ref_pagerank(A[:5,:5]))

[0.43869288 0.02171029 0.02786154 0.02171029 0.02171029 0.02786154
 0.04585394 0.39459924] float64
eigenvalues of B: [ 1.    +0.j -0.85  +0.j  0.2346+0.j -0.1283+0.j -0.    +0.j  0.    +0.j
  0.    -0.j -0.    +0.j]


The same comparison on random graphs with sinks, using all nodes and a leading $N \times N$ block, with the iterative and eigenvalue methods called one after the other on the same array:

In [14]:
rng = np.random.default_rng(13)
np.random.seed(13)
for trial in range(20):
    n = int(rng.integers(5, 60))
    adj = (rng.random((n, n)) < 0.1).astype(float)
    np.fill_diagonal(adj, 0)
    adj_before = adj.copy()
    m = int(rng.integers(3, n+1))
    d = rng.uniform(0.5, 0.9)

    G = nx.from_numpy_array(adj, create_using=nx.DiGraph)
    pr = nx.pagerank(G, alpha=d, tol=1e-10, max_iter=1000)
    p_nx = np.array([pr[i] for i in range(n)])

    assert np.allclose(iter_solve(adj, N=m, d=d), ref_pagerank(adj[:m,:m], d), atol=1e-4)
    assert np.allclose(eig_solve(adj, d=d), p_nx)
    assert np.allclose(eig_solve(adj, N=m, d=d), ref_pagerank(adj[:m,:m], d))
    assert np.allclose(iter_solve(adj, d=d), p_nx, atol=1e-4)
    assert np.array_equal(adj, adj_before)
print("All checks passed.")

All checks passed.


## Application: Ranking Sports Teams

PageRank applies to any problem with a directed graph structure, such as ranking sports teams. Each team is a node, and each game gives an edge pointing from the losing team to the winning team. Wins and losses do not cancel out: if BYU and Boise played twice and each won once, there is an edge from BYU to Boise and another from Boise to BYU. Edges are not weighted, so if Duke ever beat Harvard, whether once or 5 times, there is only one edge from Harvard to Duke.

Edges tend to lead from worse teams to better teams, so starting with some team and randomly following edges should visit better teams more often, just like PageRank. Here $d$ no longer represents boredom: it allows random jumps from one team to another, which could represent a surprise upset, or the random outcome of a game between two teams who have never played each other.

## Problem 5: Ranking NCAA teams

> **Problem 5.** By applying the PageRank algorithm to win-loss data from the 2013 NCAA basketball season, produce a comparative ranking of the teams.
> 1. The file `ncaa2013.csv` contains data on over 5000 basketball games. The first line is a header. After the header, each line represents a game and has the winning team followed by the losing team (there are no ties in basketball).
>
>    Load this file and use it to create the adjacency matrix $A$, where $A_{ij} = 1$ if team $j$ beat team $i$. Make sure to ignore the header line. You will need some way of mapping from team names to the integers and vice versa.
> 2. Use the iterative method from Problem 3 with $d = 0.7$ to find the steady state. The steady-state solution is your vector of ranks.
> 3. Return the ranks sorted from largest to smallest, and the corresponding list of teams sorted from "best" to "worst".
>
> Hints:
> 1. The code below may be helpful for processing the .csv file:
>
>    ```python
>    >>> with open('./ncaa2013.csv', 'r') as ncaafile:
>    >>>     ncaafile.readline() #reads and ignores the header line
>    >>>     for line in ncaafile:
>    >>>         teams = line.strip().split(',') #split on commas
>    >>>         print teams
>    >>> ['Middle Tenn St', 'Alabama St']
>    >>> ...
>    >>> ['Mississippi', 'Florida']
>    ```
> 2. Before creating the adjacency matrix, you can get all the unique teams by running through all the matches once and adding every team to a set. Next, count the number of unique teams and initialize $A$ to be the right size. Try using dictionaries, lists, or both to map numbers to teams and teams to numbers and fill in $A$. There is more than one right way to do this.
> 3. The function `np.argsort()` will be useful for sorting the ranks and teams.
> 4. There should be 347 teams. PageRank should predict that the top five ranked teams are Duke, Butler, Louisville, Illinois, and Indiana (in that order). Use this to check your results.

In [15]:
def team_rank(filename='ncaa2013.csv'):
    """Use iter_solve() to predict the rankings of the teams in the given
    dataset of games. The dataset should have two columns, representing
    winning and losing teams. Each row represents a game, with the winner on
    the left, loser on the right. Parse this data to create the adjacency
    matrix, and feed this into the solver to predict the team ranks.

    Parameters:
        filename (str): The name of the data file.
    Returns:
        ranks (list): The ranks of the teams from best to worst.
        teams (list): The names of the teams, also from best to worst.
    """
    teams = set()
    games = list()
    # Read the input, add the save the necessary information
    with open(filename, 'r') as myfile:
        myfile.readline()
        for line in myfile:
            game = line.strip().split(',')
            teams.add(game[0])
            teams.add(game[1])
            games.append(game)
    teams = np.array(list(teams))
    N = len(teams)
    # Create a dictionary of the teams corresponding to a numeric value
    team_dict = {name:i for i, name in enumerate(teams)}
    # Create the adjacency matrix
    A = dok_matrix((N,N))
    for g in games:
        A[team_dict[g[1]], team_dict[g[0]]] = 1
    # Calulate the rank
    p = iter_solve(A.toarray(), d=0.7)
    # Return the ranks and teams in order
    return p[np.argsort(p)][::-1], teams[np.argsort(p)][::-1]

The `ncaa2013.csv` distributed with the lab today is not the file the spec's hint 4 describes: it has 5,947 games and 630 teams (it includes many non-Division I opponents), so the spec's count of 347 teams and its top five do not apply to it. The check here is instead against NetworkX's `pagerank()` on the same games (an edge from loser to winner, $\alpha = d = 0.7$). The team order inside `team_rank()` comes from a `set`, so only exact ties at the bottom of the table can change order between runs.

In [16]:
np.random.seed(5)
ranks, teams = team_rank('ncaa2013.csv')
print(len(teams), "teams")
for i in range(10):
    print(f"{i+1:3d}. {teams[i]:20s} {ranks[i]:.5f}")

# Independent check with NetworkX.
with open('ncaa2013.csv') as f:
    f.readline()
    games = [line.strip().split(',') for line in f]
G = nx.DiGraph()
G.add_edges_from((loser, winner) for winner, loser in games)
pr = nx.pagerank(G, alpha=0.7, tol=1e-10, max_iter=1000)
print(len(games), "games;", G.number_of_nodes(), "teams in NetworkX")
assert G.number_of_nodes() == len(teams)
assert np.allclose(ranks, [pr[t] for t in teams], atol=1e-5)
top_nx = sorted(pr, key=pr.get, reverse=True)[:10]
assert list(teams[:10]) == top_nx

630 teams
  1. Wisconsin            0.00999
  2. UConn                0.00903
  3. Kentucky             0.00886
  4. Michigan State       0.00713
  5. Arizona              0.00684
  6. Florida              0.00666
  7. Baylor               0.00628
  8. Syracuse             0.00606
  9. UCLA                 0.00602
 10. Michigan             0.00597
5947 games; 630 teams in NetworkX


## NetworkX: Python package for networks

NetworkX analyzes the basic properties of a network and has its own PageRank implementation. It takes advantage of the sparse nature of these networks, so it is more efficient than the dense implementations in this lab. The spec initializes the graph of Figure 14.1 from an $n \times 2$ array of the edges in `matrix.txt`:

In [17]:
edges = np.array([[ 0, 7],
                  [ 1, 0],
                  [ 3, 0],
                  [ 3, 6],
                  [ 4, 0],
                  [ 4, 5],
                  [ 4, 6],
                  [ 5, 0],
                  [ 5, 6],
                  [ 6, 0],
                  [ 7, 0]])
G = nx.from_edgelist(edges.tolist(), create_using=nx.DiGraph())
print(dict(sorted(G.in_degree())))
print(dict(sorted(G.out_degree())))
print(G.in_edges(0))
print(G.out_edges(0))

{0: 6, 1: 0, 3: 0, 4: 0, 5: 1, 6: 3, 7: 1}
{0: 1, 1: 1, 3: 2, 4: 3, 5: 2, 6: 1, 7: 1}
[(1, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0)]
[(0, 7)]


NetworkX 2.0 and later return view objects instead of the dicts and lists the spec printed, so the degrees are converted to sorted dicts here (and `.tolist()` keeps the node labels plain integers rather than NumPy integers). The contents match the spec's, for example `{0: 6, 1: 0, 3: 0, 4: 0, 5: 1, 6: 3, 7: 1}` for the in-degrees. NetworkX's `pagerank()` takes the DiGraph directly (`alpha` is the damping factor). The spec's output was

```text
{0: 0.45323691210120065,
 1: 0.021428571428571432,
 3: 0.021428571428571432,
 4: 0.021428571428571432,
 5: 0.027500000000000004,
 6: 0.04829464285714287,
 7: 0.406682730755942}
```

In [18]:
dict(sorted(nx.pagerank(G, alpha=0.85).items()))    # alpha is the dampening factor.

{0: 0.45323691210120043,
 1: 0.021428571428571432,
 3: 0.021428571428571432,
 4: 0.021428571428571432,
 5: 0.027500000000000004,
 6: 0.04829464285714286,
 7: 0.40668273075594186}

This reproduces the spec's values, but they differ from the page ranks computed above because the spec's edge list leaves out the edge $3 \to 2$ that `matrix.txt` contains, so node 2 is missing from that graph. With the full edge list, NetworkX agrees with `eig_solve()`:

In [19]:
G_full = nx.from_edgelist(np.argwhere(to_matrix('matrix.txt', 8).toarray()).tolist(), create_using=nx.DiGraph())
pr = nx.pagerank(G_full, alpha=0.85, tol=1e-10, max_iter=1000)
print({i: round(pr[i], 8) for i in sorted(pr)})
assert np.allclose([pr[i] for i in range(8)], eig_solve(to_matrix('matrix.txt', 8).toarray()))

{0: 0.43869288, 1: 0.02171029, 2: 0.02786154, 3: 0.02171029, 4: 0.02171029, 5: 0.02786154, 6: 0.04585394, 7: 0.39459924}


## Application: Twitter Datasets

The SNAP graph library at http://snap.stanford.edu/data/index.html provides a variety of medium sized network datasets for public use, including road systems, social networks, and online communities, for those wanting to experiment further with PageRank.

> **Problem 6 (Optional).** The `twitter_combined.txt` file contains a list of edges that represent a Twitter network. To protect the privacy of users, the data has been anonymized. Each number is an ID for a user. The users in the first column represent Twitter users that follow the users in the second column.
>
> Using these edges,
> 1. Create a DiGraph object using all the edges described in `twitter_combined.txt`.
> 2. Calcualte the page ranks for this graph. The page ranks create a ranking of which users are the most "influential". Even though the results will just be numbers, remember that they represent actual Twitter users.
> 3. Analyze the in-degree and out-degree of the top 10 ranked users. What do you notice about the second-highest ranked user? Why would this user be ranked so high? HINT: Use `G.in_edges()` and `G.out_edges()` to gain further insight into this result.
> 4. Return the top 10 most influential users and their scores.

This optional problem is not part of my solutions, and `twitter_combined.txt` is not included with this lab.